# Fusion Layer — XGBoost Meta-Classifier + SHAP
## Guardian Recruit · `04_fusion_layer_shap.ipynb`

**Purpose:** Combines Stream A (BERT) and Stream B (IsolationForest) scores into a single fraud probability using XGBoost. Exports `models/fusion_xgb.json`.

> **Before running:** Ensure `Runtime → T4 GPU` is selected. You must have already run `02_nlp_stream_training.ipynb` and saved `nlp_bert.pth` to Drive.

### Pipeline
```
job posting
  ├── Stream A: BERT         → bert_score    (0.0–1.0)
  ├── Stream B: IsoForest    → outlier_score (float, lower = more anomalous)
  └── Fusion:  XGBoost       → fraud_score   (0.0–1.0)  ← final decision
```

### Meta-features fed into XGBoost
| Feature | Source |
|---|---|
| `bert_score` | Stream A — BERT fraud probability |
| `outlier_score` | Stream B — IsolationForest decision function |
| `has_company_logo` | Raw metadata |
| `has_questions` | Raw metadata |
| `desc_len` | Character length of description |

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cpu':
    print('WARNING: No GPU. BERT inference over 17k rows will be slow.')
    print('Go to Runtime → Change runtime type → T4 GPU')
else:
    print(f'GPU ready: {torch.cuda.get_device_name(0)}')

In [ ]:
!pip install transformers xgboost shap scikit-learn --quiet

In [ ]:
# ── Clone repo and wire src/ into Python path ────────────────────────────────
import os, sys
from pathlib import Path

REPO_URL  = 'https://github.com/IsaganiJulian/Guardian-Recruit-Fraud-Detection.git'
REPO_PATH = Path('/content/Guardian-Recruit-Fraud-Detection')

if not REPO_PATH.exists():
    !git clone {REPO_URL} {REPO_PATH}
else:
    !git -C {REPO_PATH} pull
    print('Repo already cloned — pulled latest.')

sys.path.insert(0, str(REPO_PATH / 'src'))
print(f'src/ added to path: {REPO_PATH / "src"}')

In [ ]:
# ── Mount Drive and set all paths ────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT   = Path('/content/drive/MyDrive/DTSC 5082- Group 12 - Guardian Recruit')
DATA_DIR     = DRIVE_ROOT / 'data' / 'preprocessed'
DRIVE_MODELS = DRIVE_ROOT / 'models'
LOCAL_MODELS = REPO_PATH / 'models'
LOCAL_MODELS.mkdir(exist_ok=True)
DRIVE_MODELS.mkdir(parents=True, exist_ok=True)

# Copy model files from Drive → local repo so src/ imports can find them
import shutil
for fname in ['nlp_bert.pth', 'outlier_forest.pkl']:
    src_path  = DRIVE_MODELS / fname
    dest_path = LOCAL_MODELS / fname
    if src_path.exists():
        shutil.copy(src_path, dest_path)
        size_mb = dest_path.stat().st_size / 1e6
        print(f'[OK] {fname} copied ({size_mb:.1f} MB)')
    else:
        print(f'[MISSING] {fname} not found on Drive — upload it in the next cell.')

print(f'\nData dir  : {DATA_DIR}')
print(f'Model dir : {LOCAL_MODELS}')

In [ ]:
# ── Upload any missing model files (run only if [MISSING] appeared above) ────
# outlier_forest.pkl lives locally at: models/outlier_forest.pkl
# nlp_bert.pth      lives locally at: models/nlp_bert.pth  (already on Drive from notebook 02)
#
# For each [MISSING] file: select it from your local machine when the picker opens.

from google.colab import files

for fname in ['outlier_forest.pkl', 'nlp_bert.pth']:
    dest = LOCAL_MODELS / fname
    drive_dest = DRIVE_MODELS / fname
    if not dest.exists() or dest.stat().st_size == 0:
        print(f'Upload {fname} now (local path: models/{fname}):')
        uploaded = files.upload()
        for up_name, data in uploaded.items():
            dest.write_bytes(data)
            drive_dest.write_bytes(data)   # save to Drive for future runs
            print(f'[OK] {fname} saved ({len(data)/1e6:.1f} MB)')
    else:
        print(f'[OK] {fname} already present — skipping upload.')

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from tqdm.auto import tqdm
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, precision_recall_curve
)
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F

import outlier_stream

print('All imports OK')

In [ ]:
# ── Load datasets ────────────────────────────────────────────────────────────
aug_path = DATA_DIR / 'FINAL_AUGMENTED_TRAINING.csv'
ori_path = DATA_DIR / 'train_clean_v1.csv'

train_df = pd.read_csv(aug_path if aug_path.exists() else ori_path)
val_df   = pd.read_csv(DATA_DIR / 'val.csv')

# Fix val.csv boolean columns
BOOL_MAP = {'t': 1, 'true': 1, 'f': 0, 'false': 0}
for col in ['has_company_logo', 'telecommuting', 'fraudulent', 'has_questions']:
    if col in val_df.columns:
        val_df[col] = (
            val_df[col].astype(str).str.lower()
            .map(BOOL_MAP)
            .fillna(pd.to_numeric(val_df[col], errors='coerce'))
        )

train_df['fraudulent'] = train_df['fraudulent'].astype(int)
val_df['fraudulent']   = val_df['fraudulent'].astype(int)

print(f'Train : {train_df.shape}  |  fraud rate: {train_df["fraudulent"].mean():.2%}')
print(f'Val   : {val_df.shape}    |  fraud rate: {val_df["fraudulent"].mean():.2%}')

---
## Step 1 — Compute Meta-Features

Run both streams on every row to build the feature matrix XGBoost will train on.
BERT inference runs in **batches of 32** for GPU efficiency — much faster than row-by-row.

Results are cached to Drive so you don't need to recompute if the cell is re-run.

In [ ]:
# ── Batched BERT inference ───────────────────────────────────────────────────
# Row-by-row inference is ~50x slower than batched — critical for 17k rows.

MODEL_NAME = 'bert-base-uncased'
MAX_LENGTH = 128
BERT_PATH  = LOCAL_MODELS / 'nlp_bert.pth'
TEXT_COLS  = ['title', 'company_profile', 'description', 'requirements']

tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

if BERT_PATH.exists() and BERT_PATH.stat().st_size > 0:
    bert_model.load_state_dict(torch.load(BERT_PATH, map_location=device, weights_only=True))
    print(f'[OK] Loaded fine-tuned weights from {BERT_PATH.name}')
else:
    print('[WARNING] nlp_bert.pth not found — using untrained weights. Scores will be ~0.5')

bert_model.to(device)
bert_model.eval()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text.strip()

def make_text(df):
    for col in TEXT_COLS:
        df[col] = df[col].fillna('').apply(clean_text)
    return (df['title'] + ' ' + df['company_profile'] + ' ' +
            df['description'] + ' ' + df['requirements']).tolist()

def compute_bert_scores(texts, batch_size=32):
    scores = []
    for i in tqdm(range(0, len(texts), batch_size), desc='BERT inference'):
        batch_texts = texts[i : i + batch_size]
        enc = tokenizer(
            batch_texts, padding='max_length', truncation=True,
            max_length=MAX_LENGTH, return_tensors='pt'
        )
        input_ids      = enc['input_ids'].to(device)
        attention_mask = enc['attention_mask'].to(device)
        with torch.no_grad():
            logits = bert_model(input_ids=input_ids, attention_mask=attention_mask).logits
            probs  = F.softmax(logits, dim=1)[:, 1]
        scores.extend(probs.cpu().numpy().tolist())
    return scores

print('BERT model ready for batched inference.')

In [ ]:
# ── Compute meta-features for training set ───────────────────────────────────
import sys, re as _re

# Ensure src/ is on path for text_signals
SRC_PATH = str(REPO_PATH / 'src')
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

PLATFORM_RE = [_re.compile(p, _re.IGNORECASE) for p in [r'\bwhatsapp\b', r'\btelegram\b', r'\bsignal\b']]

def platform_risk(text):
    return sum(1 for p in PLATFORM_RE if p.search(str(text)))

def text_perplexity_stub(text):
    return 200.0  # neutral fallback — GPT-2 too slow for bulk Colab inference

print(f'Computing meta-features for {len(train_df):,} training rows...')
train_texts       = make_text(train_df.copy())
train_bert_scores = compute_bert_scores(train_texts)

train_outlier_scores = [
    outlier_stream.anomaly_score(row)
    for _, row in tqdm(train_df.iterrows(), total=len(train_df), desc='Outlier scores')
]

combined_train = (
    train_df['title'].fillna('') + ' ' +
    train_df['company_profile'].fillna('') + ' ' +
    train_df['description'].fillna('') + ' ' +
    train_df['requirements'].fillna('')
)

meta_train = pd.DataFrame({
    'bert_score':       train_bert_scores,
    'outlier_score':    train_outlier_scores,
    'has_company_logo': train_df['has_company_logo'].fillna(0).astype(int).values,
    'has_questions':    train_df['has_questions'].fillna(0).astype(int).values,
    'desc_len':         train_df['description'].fillna('').astype(str).str.len().values,
    'domain_age_days':  [-1] * len(train_df),
    'text_perplexity':  [text_perplexity_stub(t) for t in combined_train],
    'platform_risk':    [platform_risk(t) for t in combined_train],
    'fraudulent':       train_df['fraudulent'].values,
})

TRAIN_CACHE = DRIVE_MODELS / 'meta_train_v2.csv'
meta_train.to_csv(TRAIN_CACHE, index=False)
print(f'[OK] Train meta-features saved ({meta_train.shape})')
print(meta_train.describe().round(3))

In [ ]:
# ── Compute meta-features for validation set ─────────────────────────────────
print(f'Computing meta-features for {len(val_df):,} validation rows...')
val_texts       = make_text(val_df.copy())
val_bert_scores = compute_bert_scores(val_texts)

val_outlier_scores = [
    outlier_stream.anomaly_score(row)
    for _, row in tqdm(val_df.iterrows(), total=len(val_df), desc='Outlier scores')
]

combined_val = (
    val_df['title'].fillna('') + ' ' +
    val_df['company_profile'].fillna('') + ' ' +
    val_df['description'].fillna('') + ' ' +
    val_df['requirements'].fillna('')
)

meta_val = pd.DataFrame({
    'bert_score':       val_bert_scores,
    'outlier_score':    val_outlier_scores,
    'has_company_logo': val_df['has_company_logo'].fillna(0).astype(int).values,
    'has_questions':    val_df['has_questions'].fillna(0).astype(int).values,
    'desc_len':         val_df['description'].fillna('').astype(str).str.len().values,
    'domain_age_days':  [-1] * len(val_df),
    'text_perplexity':  [text_perplexity_stub(t) for t in combined_val],
    'platform_risk':    [platform_risk(t) for t in combined_val],
    'fraudulent':       val_df['fraudulent'].values,
})

VAL_CACHE = DRIVE_MODELS / 'meta_val_v2.csv'
meta_val.to_csv(VAL_CACHE, index=False)
print(f'[OK] Val meta-features saved ({meta_val.shape})')

---
## Step 2 — Train XGBoost Fusion Model

In [ ]:
META_FEATURES = [
    'bert_score', 'outlier_score', 'has_company_logo', 'has_questions', 'desc_len',
    'domain_age_days', 'text_perplexity', 'platform_risk',
]

X_train = meta_train[META_FEATURES]
y_train = meta_train['fraudulent']
X_val   = meta_val[META_FEATURES]
y_val   = meta_val['fraudulent']

fraud_count  = y_train.sum()
legit_count  = len(y_train) - fraud_count
scale_weight = legit_count / max(fraud_count, 1)
print(f'Train fraud: {fraud_count:,}  |  legit: {legit_count:,}  |  scale_pos_weight: {scale_weight:.2f}')

fusion_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_weight,
    eval_metric='logloss',
    random_state=42,
)

fusion_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50,
)
print('XGBoost training complete.')

In [ ]:
# ── Save to Drive and local repo ─────────────────────────────────────────────
FUSION_DRIVE = DRIVE_MODELS / 'fusion_xgb.json'
FUSION_LOCAL = LOCAL_MODELS / 'fusion_xgb.json'

fusion_model.save_model(FUSION_DRIVE)
fusion_model.save_model(FUSION_LOCAL)
print(f'[OK] fusion_xgb.json saved to Drive and local repo')

---
## Step 3 — Evaluate on Validation Set

In [ ]:
THRESHOLD = 0.3   # matches nlp_stream optimised threshold

all_probs = fusion_model.predict_proba(X_val)[:, 1]
all_preds = (all_probs >= THRESHOLD).astype(int)
all_true  = y_val.values

print('=' * 55)
print('FUSION LAYER — Classification Report (val.csv)')
print('=' * 55)
print(classification_report(all_true, all_preds, target_names=['Legitimate', 'Fraudulent'], zero_division=0))
print(f'ROC-AUC: {roc_auc_score(all_true, all_probs):.4f}')

In [ ]:
# ── Stream-by-stream comparison ───────────────────────────────────────────────
from sklearn.metrics import f1_score, precision_score, recall_score

bert_preds    = (meta_val['bert_score'] >= THRESHOLD).astype(int)
fusion_preds  = all_preds

comparison = pd.DataFrame({
    'Model':     ['Stream A (BERT)', 'Fusion (XGBoost)'],
    'Precision': [
        precision_score(all_true, bert_preds,   zero_division=0),
        precision_score(all_true, fusion_preds, zero_division=0),
    ],
    'Recall': [
        recall_score(all_true, bert_preds),
        recall_score(all_true, fusion_preds),
    ],
    'F1': [
        f1_score(all_true, bert_preds,   zero_division=0),
        f1_score(all_true, fusion_preds, zero_division=0),
    ],
}).round(4)

print(comparison.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(3)
w = 0.35
vals_bert   = comparison.iloc[0][['Precision','Recall','F1']].values
vals_fusion = comparison.iloc[1][['Precision','Recall','F1']].values
ax.bar(x - w/2, vals_bert,   w, label='Stream A (BERT)',   color='steelblue')
ax.bar(x + w/2, vals_fusion, w, label='Fusion (XGBoost)',  color='tomato')
ax.set_xticks(x)
ax.set_xticklabels(['Precision', 'Recall', 'F1'])
ax.set_ylim(0, 1.05)
ax.set_title('Stream A vs Fusion Layer — Fraud Class Metrics')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
cm = confusion_matrix(all_true, all_preds, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Legitimate', 'Fraudulent']).plot(
    ax=axes[0], colorbar=False, cmap='Reds')
axes[0].set_title('Confusion Matrix — Fusion Layer (Validation Set)')

axes[1].axis('off')
table_data = [
    ['Metric', 'Count', 'Meaning'],
    ['True Positives (TP)',  tp, 'Fraud correctly flagged'],
    ['True Negatives (TN)',  tn, 'Legit correctly cleared'],
    ['False Positives (FP)', fp, 'Legit wrongly flagged'],
    ['False Negatives (FN)', fn, 'Fraud missed'],
    ['Precision', f'{tp/(tp+fp+1e-9):.3f}', ''],
    ['Recall',    f'{tp/(tp+fn+1e-9):.3f}', ''],
]
t = axes[1].table(cellText=table_data[1:], colLabels=table_data[0], loc='center', cellLoc='left')
t.auto_set_font_size(True)
t.scale(1, 1.6)
plt.suptitle('Fusion Layer Confusion Matrix Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'False Negatives: {fn}  (BERT alone: 32)')
print(f'False Positives: {fp}  (BERT alone: 6)')

In [ ]:
# ── Threshold sweep ───────────────────────────────────────────────────────────
rows = []
for thresh in [0.1, 0.2, 0.3, 0.4, 0.5]:
    preds_t = (all_probs >= thresh).astype(int)
    cm_t = confusion_matrix(all_true, preds_t, labels=[0, 1])
    tn_t, fp_t, fn_t, tp_t = cm_t.ravel()
    prec = tp_t / (tp_t + fp_t + 1e-9)
    rec  = tp_t / (tp_t + fn_t + 1e-9)
    f1   = 2 * prec * rec / (prec + rec + 1e-9)
    rows.append({'Threshold': thresh, 'TP': tp_t, 'FP': fp_t, 'FN': fn_t,
                 'Precision': round(prec, 3), 'Recall': round(rec, 3), 'F1': round(f1, 3)})

print('Fusion Layer — FP vs FN Tradeoff:')
print(pd.DataFrame(rows).to_string(index=False))

---
## Step 4 — SHAP Explainability

SHAP (SHapley Additive exPlanations) shows **which features drove each prediction**.
This answers: *why did the model flag this posting as fraud?*

In [ ]:
explainer   = shap.TreeExplainer(fusion_model)
shap_raw    = explainer.shap_values(X_val)

# SHAP API differs by version:
# older SHAP → list of two arrays [legit_shap, fraud_shap]
# newer SHAP → single 2D array (fraud class only for binary)
shap_vals = shap_raw[1] if isinstance(shap_raw, list) else shap_raw

plt.figure(figsize=(8, 5))
shap.summary_plot(shap_vals, X_val, feature_names=META_FEATURES, show=False)
plt.title('SHAP Feature Importance — Fusion Layer')
plt.tight_layout()
plt.show()

In [ ]:
# Mean absolute SHAP values — overall feature importance ranking
mean_shap = np.abs(shap_vals).mean(axis=0)
importance_df = pd.DataFrame({'Feature': META_FEATURES, 'Mean |SHAP|': mean_shap})
importance_df = importance_df.sort_values('Mean |SHAP|', ascending=True)

plt.figure(figsize=(7, 4))
plt.barh(importance_df['Feature'], importance_df['Mean |SHAP|'], color='steelblue')
plt.xlabel('Mean |SHAP value|')
plt.title('Feature Importance — Fusion Layer (SHAP)')
plt.tight_layout()
plt.show()

print(importance_df.sort_values('Mean |SHAP|', ascending=False).to_string(index=False))

---
## Summary

| Deliverable | Detail |
|---|---|
| Meta-features computed | `bert_score`, `outlier_score`, `has_company_logo`, `has_questions`, `desc_len` |
| Fusion model trained | XGBoost, 300 estimators, `scale_pos_weight` for class imbalance |
| Model saved | `models/fusion_xgb.json` (Drive + local repo) |
| Explainability | SHAP summary + bar chart |

**Using the pipeline locally:**
```python
from main import score

result = score({
    'title': 'Work From Home — Unlimited Earnings',
    'description': 'No experience needed.',
    'has_company_logo': 0,
    'has_questions': 0,
})
print(result['label'])       # FRAUD or LEGITIMATE
print(result['fraud_score']) # 0.0 – 1.0
```